<a href="https://colab.research.google.com/github/somendrew/LangGraph_tutorial/blob/main/Projects/P1_chatbot_with_memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -qqq langchain-openai langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 5.1 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
api_key = userdata.get('api_key')

In [12]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages  # better than operator.add

#creating state
class ChatbotState(TypedDict):
  messages : Annotated[list,add_messages]

#importing LLM
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(api_key = api_key,max_tokens=100)

#creating node
def chatbot(state:ChatbotState):
  response = llm.invoke(state['messages'])
  return {'messages': [response.content]}

#creating graph
from langgraph.graph import StateGraph, START, END

graph = StateGraph(ChatbotState)
graph.add_node('chatbot',chatbot)

graph.add_edge(START, 'chatbot')
graph.add_edge('chatbot',END)

from langgraph.checkpoint.memory import MemorySaver
memory = MemorySaver()

app = graph.compile(checkpointer = memory)


In [13]:
config = {"configurable": {"thread_id": "1"}}

while True:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "quit"]:
        break
    result = app.invoke(
        {"messages": [{"role": "user", "content": user_input}]},
        config=config
    )
    print("Bot:", result['messages'][-1].content)

You: hi
Bot: Hello! How can I assist you today?
You: do u know what  is jaadu?
Bot: "Jaadu" is a term that can have different meanings depending on the context or culture. In Hindi or Urdu, "Jaadu" generally refers to magic or sorcery. It can also mean magic tricks or illusions performed for entertainment purposes. Let me know if you need more information on this topic.
You: Its a movie
Bot: I see! "Jaadu" is a 1990 Bollywood film directed by Arshad Khan. The movie stars Amitabh Bachchan, Jayapradha, Aditya Pancholi, and Amrita Singh. It is a drama film that revolves around the theme of love and relationships. If you have any more questions about the movie or its plot, feel free to ask!
You: what is my name?
Bot: I'm sorry, but I don't have access to your personal information, including your name. If there's anything else you'd like to know or discuss, feel free to ask.
You: my name is somendra
Bot: Hello, Somendra! How can I assist you today?
You: what is my name and age
Bot: Your nam